# Leitura de Parquet com PySpark

Notebook simples para ler e explorar arquivos Parquet usando Apache Spark.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [1]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Ler Arquivo Parquet

Carregar um arquivo Parquet em um DataFrame Spark. Altere o caminho abaixo para o seu arquivo.

In [2]:
# Altere o caminho para o seu arquivo Parquet
caminho_parquet = "quickbooks_data/daily_incremental/Account_20260325043100.parquet"

df = spark.read.parquet(caminho_parquet)
print(f"Arquivo carregado com sucesso! Registros: {df.count()}")

Arquivo carregado com sucesso! Registros: 10


## 3. Transformar JSON em Tabela

A coluna do Parquet contém dados em formato JSON. Vamos parsear o JSON e expandir em colunas.

In [3]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Identificar o nome da coluna com JSON
col_json = df.columns[0]
print(f"Coluna JSON: {col_json}")

# Pegar uma amostra para inferir o schema
amostra = df.select(col_json).first()[0]
json_schema = schema_of_json(amostra)

# Parsear o JSON e expandir em colunas
df_parsed = df.withColumn("parsed", from_json(col(col_json), json_schema)) \
              .select("parsed.*")

print(f"Colunas resultantes: {df_parsed.columns}")
df_parsed.printSchema()

Coluna JSON: raw_value
Colunas resultantes: ['AccountSubType', 'AccountType', 'AcctNum', 'Active', 'Classification', 'CurrencyRef', 'CurrentBalance', 'CurrentBalanceWithSubAccounts', 'Description', 'FullyQualifiedName', 'Id', 'MetaData', 'Name', 'ParentRef', 'SubAccount', 'SyncToken', 'TaxCodeRef', 'domain', 'sparse']
root
 |-- AccountSubType: string (nullable = true)
 |-- AccountType: string (nullable = true)
 |-- AcctNum: string (nullable = true)
 |-- Active: string (nullable = true)
 |-- Classification: string (nullable = true)
 |-- CurrencyRef: string (nullable = true)
 |-- CurrentBalance: string (nullable = true)
 |-- CurrentBalanceWithSubAccounts: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- FullyQualifiedName: string (nullable = true)
 |-- Id: string (nullable = true)
 |-- MetaData: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- ParentRef: string (nullable = true)
 |-- SubAccount: string (nullable = true)
 |-- SyncToken: string (nul

In [4]:
# Visualizar os dados parseados
df_parsed.show(truncate=False)

+----------------+----------------+-------+------+--------------+---------------------------------------------+--------------+-----------------------------+-------------------------+--------------------+--------+-------------------------------------------------------------------------------------------------+--------------------+---------+----------+---------+-------------------------------+------+------+
|AccountSubType  |AccountType     |AcctNum|Active|Classification|CurrencyRef                                  |CurrentBalance|CurrentBalanceWithSubAccounts|Description              |FullyQualifiedName  |Id      |MetaData                                                                                         |Name                |ParentRef|SubAccount|SyncToken|TaxCodeRef                     |domain|sparse|
+----------------+----------------+-------+------+--------------+---------------------------------------------+--------------+-----------------------------+-------------------------+

In [5]:
# Estatísticas descritivas
df_parsed.describe().show()

+-------+----------------+----------------+------------------+------+--------------+--------------------+-----------------+-----------------------------+--------------------+-------------------+--------+--------------------+-------------------+---------+----------+------------------+--------------------+------+------+
|summary|  AccountSubType|     AccountType|           AcctNum|Active|Classification|         CurrencyRef|   CurrentBalance|CurrentBalanceWithSubAccounts|         Description| FullyQualifiedName|      Id|            MetaData|               Name|ParentRef|SubAccount|         SyncToken|          TaxCodeRef|domain|sparse|
+-------+----------------+----------------+------------------+------+--------------+--------------------+-----------------+-----------------------------+--------------------+-------------------+--------+--------------------+-------------------+---------+----------+------------------+--------------------+------+------+
|  count|              10|              

## 4. Executar Consultas SQL no Parquet

Registrar o DataFrame como view temporária e executar SQL.

In [6]:
# Registrar como view temporária
df_parsed.createOrReplaceTempView("dados")

# Executar consulta SQL
resultado = spark.sql("SELECT * FROM dados LIMIT 10")
resultado.show()

+----------------+----------------+-------+------+--------------+--------------------+--------------+-----------------------------+--------------------+--------------------+--------+--------------------+--------------------+---------+----------+---------+--------------------+------+------+
|  AccountSubType|     AccountType|AcctNum|Active|Classification|         CurrencyRef|CurrentBalance|CurrentBalanceWithSubAccounts|         Description|  FullyQualifiedName|      Id|            MetaData|                Name|ParentRef|SubAccount|SyncToken|          TaxCodeRef|domain|sparse|
+----------------+----------------+-------+------+--------------+--------------------+--------------+-----------------------------+--------------------+--------------------+--------+--------------------+--------------------+---------+----------+---------+--------------------+------+------+
|  OfficeExpenses|         Expense|   1101|  true|       Expense|{"value":"USD","n...|       1125.00|                      1330

In [7]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
